In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/atp_matches_merged.csv")

## Handle Missing Values

In [3]:
missing_values = df.isnull().sum()

columns_with_nan = missing_values[missing_values > 0]

print("Columns with NaN values and their counts:")
print(columns_with_nan)

Columns with NaN values and their counts:
winner_seed           11990
winner_entry          18620
winner_ht                37
loser_seed            16324
loser_entry           16632
loser_ht                118
loser_age                 1
minutes                1501
w_ace                   149
w_df                    149
w_svpt                  149
w_1stIn                 149
w_1stWon                149
w_2ndWon                149
w_SvGms                 148
w_bpSaved               149
w_bpFaced               149
l_ace                   149
l_df                    149
l_svpt                  149
l_1stIn                 149
l_1stWon                149
l_2ndWon                149
l_SvGms                 148
l_bpSaved               149
l_bpFaced               149
winner_rank              13
winner_rank_points       13
loser_rank               58
loser_rank_points        58
W1                      141
W2                      314
W3                    11368
W4                    19835
W5    

### For height we will get it if this player had it somewhere else in df, if not fill with mean

In [4]:
missing_winner_ht = df[df['winner_ht'].isnull()]['winner_id'].unique()
missing_loser_ht = df[df['loser_ht'].isnull()]['loser_id'].unique()
players_missing_ht = np.unique(np.concatenate([missing_winner_ht, missing_loser_ht]))

player_heights = {}
for player in players_missing_ht:
    height = df.loc[df['winner_id'] == player, 'winner_ht'].dropna()
    if height.empty:
        height = df.loc[df['loser_id'] == player, 'loser_ht'].dropna()
    if not height.empty:
        player_heights[player] = height.iloc[0]

df['winner_ht'] = df['winner_ht'].fillna(df['winner_id'].map(player_heights))
df['loser_ht'] = df['loser_ht'].fillna(df['loser_id'].map(player_heights))
df['winner_ht'] = df['winner_ht'].fillna(df['winner_ht'].mean())
df['loser_ht'] = df['loser_ht'].fillna(df['loser_ht'].mean())

### Sets won and minutes we can fill with 0 if match was not completed, which explains NaNs in these columns

In [5]:
df['Comment'].unique()


array(['Completed', 'Retired', 'Walkover', 'Awarded', 'Sched',
       'Disqualified', 'Rrtired'], dtype=object)

In [6]:
df['Comment'] = df['Comment'].replace({'Rrtired': 'Retired', 'Def.': 'Default'})

In [7]:
df.loc[df['match_id'] =='Estoril_2022_202104_126207', 'Wsets'] = 2
df.loc[df['match_id'] =='Metz_2019_104542_105379', ['W2','L2', 'W3','L3', 'Wsets']] = [7, 6, 6, 3, 2]
df.loc[df['match_id'] =='Nottingham_2015_104797_104571', 'Comment'] = 'Retired'

In [8]:
columns_to_update = ['Wsets','Lsets', 'minutes']
df.loc[df['Comment'].isin(['Disqualified','Retired', 'Walkover', 'Awarded', 'Default']), columns_to_update]=df.loc[df['Comment'].isin(['Disqualified','Retired', 'Walkover', 'Awarded', 'Default']), columns_to_update].fillna(0)

### Missing Minutes for completed matches we will fill with mean

In [9]:
average_minutes = df[df['Comment'] == 'Completed'].groupby('best_of')['minutes'].mean().round()
df.loc[(df['Comment'] == 'Completed') & (df['minutes'].isnull()), 'minutes'] = df['best_of'].map(average_minutes)

### Entry one hot, seed to binary

In [10]:
ENTRY_TYPES = ['WC', 'Q', 'LL']
df['winner_entry'] = df['winner_entry'].where(df['winner_entry'].isin(ENTRY_TYPES))
df['loser_entry'] = df['loser_entry'].where(df['loser_entry'].isin(ENTRY_TYPES))
df = pd.get_dummies(df, columns=['winner_entry', 'loser_entry'], prefix=['winner_entry', 'loser_entry'])

df['winner_is_seeded'] = df['winner_seed'].notna().astype(int)
df['loser_is_seeded'] = df['loser_seed'].notna().astype(int)
df = df.drop(columns=['winner_seed', 'loser_seed'])

### Nan for Rank and Rank points mean player without ranking, we'll set rank to 2000 and points to 0

In [11]:
# Rank: fill missing with 2000 (unranked), points with 0
df['winner_rank'] = df['winner_rank'].fillna(2000)
df['loser_rank'] = df['loser_rank'].fillna(2000)
df['winner_rank_points'] = df['winner_rank_points'].fillna(0)
df['loser_rank_points'] = df['loser_rank_points'].fillna(0)

# Age: fill with mean
df['winner_age'] = df['winner_age'].fillna(df['winner_age'].mean())
df['loser_age'] = df['loser_age'].fillna(df['loser_age'].mean())

# Hand: fill with R (right-handed)
df['winner_hand'] = df['winner_hand'].fillna("R")
df['loser_hand'] = df['loser_hand'].fillna("R")

### Age fill with mean

In [12]:
df['winner_age'] = df['winner_age'].fillna(np.mean(df['winner_age']))
df['loser_age'] = df['loser_age'].fillna(np.mean(df['loser_age']))

In [13]:
missing_values = df.isnull().sum()

columns_with_nan = missing_values[missing_values > 0]

print("Columns with NaN values and their counts:")
print(columns_with_nan)

Columns with NaN values and their counts:
w_ace          149
w_df           149
w_svpt         149
w_1stIn        149
w_1stWon       149
w_2ndWon       149
w_SvGms        148
w_bpSaved      149
w_bpFaced      149
l_ace          149
l_df           149
l_svpt         149
l_1stIn        149
l_1stWon       149
l_2ndWon       149
l_SvGms        148
l_bpSaved      149
l_bpFaced      149
W1             141
W2             314
W3           11367
W4           19835
W5           21260
L1             138
L2             314
L3           11367
L4           19835
L5           21260
AvgW            19
AvgL            19
dtype: int64


### We're left with NaN values for match stat values, which we will leave because we will not use these anyways, Nan values for score columns which are natural when the match was shorter than full 5 sets (most matches) and some NaN values for betting odds

### We can also check that NaN values for score columns for 1st set are due to match not being completed

In [14]:
print(df[(pd.isna(df['W1']))]['Comment'].unique())
print(df[(pd.isna(df['L1']))]['Comment'].unique())

['Walkover' 'Retired']
['Walkover' 'Retired']


### We will also rename some columns to be more appropriate

In [15]:
df = df.rename(columns={'Tournament': 'tournament_name', 'Court': 'indoor_or_outdoor', 'Series': 'tournament_level', 'tourney_location': 'tournament_location', 'tourney_date': 'tournament_date', 'tourney_id': 'tournament_id'})
df = df.drop(columns={'tourney_level'})

In [16]:
df.to_csv("../data/atp_matches_preprocessed.csv", index=False)